In [0]:
dbutils.widgets.text("stg_name","nagiligaristg")
dbutils.widgets.text("container_name","gizmobox")
dbutils.widgets.text("connection","nagiligaristgcredential")


In [0]:
storage_account = dbutils.widgets.get("stg_name")
container = dbutils.widgets.get("container_name")
connection = dbutils.widgets.get("connection")

In [0]:
%fs ls

In [0]:
display(dbutils.fs.ls(f'abfss://{container}@{storage_account}.dfs.core.windows.net'))

In [0]:
%py
spark.sql(f"""
    CREATE EXTERNAL LOCATION IF NOT EXISTS adbstgus2_ext_adbws
    URL 'abfss://{container}@{storage_account}.dfs.core.windows.net'
    WITH (STORAGE CREDENTIAL `{connection}`)
    COMMENT 'External loaction for gizmobox'
    """)

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%py
spark.sql(f"""CREATE CATALOG IF NOT EXISTS gizmobox
   MANAGED LOCATION 'abfss://{container}@{storage_account}.dfs.core.windows.net/'
   COMMENT 'This is the catalog for GizmoBox Data Lakehouse';
   """)


In [0]:
%sql
SELECT current_catalog();

In [0]:
%sql
USE CATALOG gizmobox;

In [0]:
%py
spark.sql(f"""CREATE SCHEMA IF NOT EXISTS gizmobox.landing
    MANAGED LOCATION 'abfss://{container}@{storage_account}.dfs.core.windows.net/landing'
    """)

In [0]:
%sql
SHOW SCHEMAS;

In [0]:
%py
spark.sql(f"""CREATE SCHEMA IF NOT EXISTS gizmobox.bronze
    MANAGED LOCATION 'abfss://{container}@{storage_account}.dfs.core.windows.net/bronze';
    """)

In [0]:
spark.sql(f"""
          CREATE SCHEMA IF NOT EXISTS gizmobox.silver
    MANAGED LOCATION 'abfss://{container}@{storage_account}.dfs.core.windows.net/silver';""")

In [0]:
spark.sql(f"""
          CREATE SCHEMA IF NOT EXISTS gizmobox.gold
    MANAGED LOCATION 'abfss://{container}@{storage_account}.dfs.core.windows.net/gold';
    """)

In [0]:
%sql
USE CATALOG gizmobox;
USE SCHEMA landing;
CREATE EXTERNAL VOLUME gizmobox.landing.operational_data
LOCATION 'abfss://gizmobox@adbstgus2.dfs.core.windows.net/landing/operational_data/';

In [0]:
%fs ls '/Volumes/gizmobox/landing/operational_data'

In [0]:
%sql
DROP CATALOG IF EXISTS gizmobox CASCADE